# Reject Option Classification (ROC) on Adult, German and COMPAS

## Step 1: Load the three datasets

In [1]:
# pip install aif360==0.6.1 "pandas<3" scikit-learn notebook
import logging, os, urllib.request, warnings
logging.getLogger().setLevel(logging.ERROR)   # hide AIF360 messages about optional packages
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import aif360
from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.postprocessing import RejectOptionClassification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42

# AIF360 reads the raw files from its own data folder
RAW = os.path.join(os.path.dirname(aif360.__file__), 'data', 'raw')
UCI = 'https://archive.ics.uci.edu/ml/machine-learning-databases/'
FILES = {'adult/adult.data': UCI + 'adult/adult.data',
         'adult/adult.test': UCI + 'adult/adult.test',
         'german/german.data': UCI + 'statlog/german/german.data',
         'compas/compas-scores-two-years.csv':
             'https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv'}
for path, url in FILES.items():
    if not os.path.exists(os.path.join(RAW, path)):
        urllib.request.urlretrieve(url, os.path.join(RAW, path))

# sensitive attribute per dataset (1 = privileged: male, older than 25, Caucasian)
DATA = {'Adult': (AdultDataset(), 'sex'), 'German': (GermanDataset(), 'age'), 'COMPAS': (CompasDataset(), 'race')}
for name, (ds, attr) in DATA.items():
    ds.labels = (ds.labels == ds.favorable_label).astype(float)   # 1 = good outcome, 0 = bad outcome
    ds.favorable_label, ds.unfavorable_label = 1.0, 0.0
    print(name, ds.features.shape)

Adult (45222, 98)
German (1000, 58)
COMPAS (6167, 401)


## Step 2: Fairness of the original labels

In [2]:
def fairness(ds, attr):
    # DP_Diff, DIR and consistency of the labels in ds
    m = BinaryLabelDatasetMetric(ds, unprivileged_groups=[{attr: 0}], privileged_groups=[{attr: 1}])
    return {'DP_Diff': m.statistical_parity_difference(), 'DIR': m.disparate_impact(),
            'Consistency': float(np.ravel(m.consistency(n_neighbors=5))[0])}

results = []
for name, (ds, attr) in DATA.items():
    full = ds.copy(deepcopy=True)
    full.features = StandardScaler().fit_transform(ds.features)   # scaled, so no column dominates consistency
    results.append({'Dataset': name, 'Stage': 'Original labels (before)', **fairness(full, attr)})
pd.DataFrame(results).round(3)

,Dataset,Stage,DP_Diff,DIR,Consistency
0,Adult,Original labels (before),-0.199,0.363,0.829
1,German,Original labels (before),-0.149,0.795,0.740
2,COMPAS,Original labels (before),-0.097,0.840,0.668


## Step 3: Stratified split (50% train, 20% validation, 30% test)

In [3]:
SPLIT = {}
for name, (ds, attr) in DATA.items():
    label = ds.labels.ravel().astype(int)
    group = ds.protected_attributes[:, ds.protected_attribute_names.index(attr)].astype(int)
    strata = group * 2 + label   # stratify on group x label
    rest, test = train_test_split(np.arange(len(label)), test_size=0.30, stratify=strata, random_state=SEED)
    train, val = train_test_split(rest, test_size=0.20 / 0.70, stratify=strata[rest], random_state=SEED)
    SPLIT[name] = (train, val, test)
    print(name, 'train', len(train), 'validation', len(val), 'test', len(test))

Adult train 22610 validation 9045 test 13567
German train 499 validation 201 test 300
COMPAS train 3082 validation 1234 test 1851


## Step 4: Train the model and apply ROC (post-processing)

In [4]:
# threshold fixed at 0.5; 50 widths of the critical region are tried on the validation part
ROC_SETTINGS = dict(low_class_thresh=0.5, high_class_thresh=0.51, num_class_thresh=1, num_ROC_margin=50,
                    metric_name='Statistical parity difference', metric_lb=-0.05, metric_ub=0.05)

TEST = {}
for name, (ds, attr) in DATA.items():
    train, val, test = SPLIT[name]
    scaler = StandardScaler().fit(ds.features[train])
    model = LogisticRegression(max_iter=2000).fit(scaler.transform(ds.features[train]), ds.labels[train].ravel())

    def true_and_predicted(rows):
        true = ds.subset(rows)                    # real labels
        true.features = scaler.transform(true.features)
        pred = true.copy(deepcopy=True)           # model predictions
        pred.scores = model.predict_proba(true.features)[:, [1]]
        pred.labels = (pred.scores > 0.5).astype(float)
        return true, pred

    val_true, val_pred = true_and_predicted(val)
    test_true, test_pred = true_and_predicted(test)

    roc = RejectOptionClassification(unprivileged_groups=[{attr: 0}], privileged_groups=[{attr: 1}], **ROC_SETTINGS)
    roc.fit(val_true, val_pred)          # choose theta on the validation part
    test_roc = roc.predict(test_pred)    # change the test predictions
    TEST[name] = (test_true, test_pred, test_roc)
    print(name, 'theta =', round(0.5 + roc.ROC_margin, 2))

Adult theta = 0.68
German theta = 0.59
COMPAS theta = 0.53


## Step 5: Fairness and accuracy on the test set

In [5]:
# compare the test predictions with and without ROC
for name, (ds, attr) in DATA.items():
    test_true, test_pred, test_roc = TEST[name]
    for stage, pred in [('Test predictions without ROC', test_pred), ('Test predictions with ROC (after)', test_roc)]:
        # fairness of the predictions, and accuracy against the real labels
        results.append({'Dataset': name, 'Stage': stage, **fairness(pred, attr),
                        'Accuracy': accuracy_score(test_true.labels.ravel(), pred.labels.ravel())})

## Step 6: Before and after mitigation

In [6]:
# all results in one table, sorted per dataset (Table 3 of the report)
table = pd.DataFrame(results)
table['Dataset'] = pd.Categorical(table['Dataset'], categories=list(DATA), ordered=True)
table.sort_values('Dataset', kind='stable').set_index(['Dataset', 'Stage']).round(3)

DP_Diff    DIR  Consistency  \
Dataset Stage                                                            
Adult   Original labels (before)            -0.199  0.363        0.829   
        Test predictions without ROC        -0.186  0.290        0.945   
        Test predictions with ROC (after)   -0.042  0.733        0.955   
German  Original labels (before)            -0.149  0.795        0.740   
        Test predictions without ROC        -0.068  0.920        0.846   
        Test predictions with ROC (after)   -0.022  0.972        0.824   
COMPAS  Original labels (before)            -0.097  0.840        0.668   
        Test predictions without ROC        -0.169  0.764        0.858   
        Test predictions with ROC (after)   -0.071  0.893        0.855   

                                           Accuracy  
Dataset Stage                                        
Adult   Original labels (before)                NaN  
        Test predictions without ROC          0.850  
        Test predictions with ROC (after)     0.834  
German  Original labels (before)                NaN  
        Test predictions without ROC          0.773  
        Test predictions with ROC (after)     0.770  
COMPAS  Original labels (before)                NaN  
        Test predictions without ROC          0.671  
        Test predictions with ROC (after)     0.663